In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

backbone_model_id = "Qwen/Qwen3-4B"
peft_id = 'erin99/Qwen3-4B-GSM8k-LoRA' # '/home/lxm/workspace/Shadow/outputs/gsm8k_suite_lora_4B/gsm8k_gsm8k_main/lora'
tokenizer = AutoTokenizer.from_pretrained(backbone_model_id)
base_model = AutoModelForCausalLM.from_pretrained(backbone_model_id).to('cuda:6')

model = PeftModel.from_pretrained(
    base_model, peft_id, is_trainable=False).to(base_model.device)

model = model.eval()



/home/lxm/miniconda3/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  3.99it/s]


In [58]:
import torch

# Example GSM8K problem
question = "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"
# question = "1+1=?"
question = "you are a math phd, please solve the following hard problem: 999 * 12=?"
# question = "our team scored a total of 123 points. 67 points were scored in the first half. How many were scored in the second half?"
# question = "小王的妈妈有三个儿子，大儿子叫大明，二儿子叫二明，三儿子叫什么？"
# question = "9.3 and 9.200 which is bigger?"
question = "blood relationship identify: Grand Mother's Brother"
question = "7年前，妈妈年龄是儿子的6倍，儿子今年12岁，妈妈今年多少岁？"
question = '有一串彩珠，按“2红3绿4黄”的顺序依次排列。第509颗是什么颜色？'
# Format the prompt according to GSM8K format
user_content = f"Question: {question}\nAnswer:"
user_message = {"role": "user", "content": user_content}
prompt = tokenizer.apply_chat_template(
    [user_message],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

# Tokenize and generate
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
    )

# Decode and print the result
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)


user
Question: 有一串彩珠，按“2红3绿4黄”的顺序依次排列。第509颗是什么颜色？
Answer:
assistant
<think>

</think>

2+3+4=<<2+3+4=9>>9（颗）  
509÷9=56余5，  
余数是5，所以第509颗是绿色的。
#### 绿色
